In [ ]:
import os
import subprocess
from pathlib import Path

import anndata as ad
import scanpy as sc

from utils import DATA_DIR, RESULT_DIR, SHELL, activate_env_cmd, remove_tick_and_label

In [ ]:
input = DATA_DIR / "ABC_atlas" / "Zhuang-ABCA-1.h5ad"

results_abc = RESULT_DIR / "ABC_atlas"

script_dir = Path("batch_integration")

seed = 42
n_components = 50

In [ ]:
results_abc.mkdir(exist_ok=True, parents=True)

In [ ]:
env = os.environ.copy()
env["MPLBACKEND"] = "pdf"  # avoid matplotlib backend issues

# Preprocess

In [ ]:
adata = ad.read_h5ad(input)

layer = "scaled"
adata.layers[layer] = adata.X.copy()
sc.pp.log1p(adata, layer=layer)
sc.pp.pca(adata, n_comps=n_components, zero_center=True, random_state=seed, layer=layer)

adata.write_h5ad(input)

# Batch correction

BBKNN selects n-neighbors per batch, given that we have 147 sections this approach seems less suitable.

In [ ]:
methods = [
    "no_batch_integration",
    "harmony",
    "scvi",
]

In [ ]:
for method in methods:
    out_file = results_abc / f"{method}.npz"
    script_path = script_dir / f"{method}.py"

    conda_cmd = activate_env_cmd(method)
    cmd = f"{script_path} {input} {out_file}"

    # use same number of components for scVI as PCs for harmony
    if method == "scvi":
        cmd += f" --n_comp {n_components}"

    subprocess.run(f"{conda_cmd} && {cmd}", shell=True, executable=SHELL, env=env)

# Clustering

In [ ]:
import time

from scipy.sparse import load_npz
from spatialleiden import distance2connectivity, spatialleiden

In [ ]:
adata = ad.read_h5ad(input)

sq.gr.spatial_neighbors(
    adata, library_key=adata.uns["batch_keys"][0], coord_type="generic", n_neighs=10
)

adata.obsp["spatial_connectivities"] = distance2connectivity(
    adata.obsp["spatial_distances"]
)

In [ ]:
for file in results_abc.glob("*.npz"):
    method = file.stem

    print(f"Processing {method}")

    adata.obsp["connectivities"] = load_npz(file)

    start = time.time()
    spatialleiden(
        adata,
        resolution=(1, 1),
        directed=(False, True),
        layer_ratio=2,
        n_iterations=3,
    )
    end = time.time()
    print(f"Time: {(end - start) / 60:.2f} min")

    columns = ["spatialleiden"] + list(adata.uns["batch_keys"])

    labels = adata.obs[columns]
    labels.loc[:, ["x", "y", "z"]] = adata.obsm["spatial"]
    labels.to_parquet(results_abc / f"{method}.parquet")

# Plotting

In [ ]:
import pandas as pd
import seaborn as sns

scatter_kwargs = {
    "s": 1,
    "linewidth": 0,
}

In [ ]:
import matplotlib.pyplot as plt

# Don't show figure
_ = plt.ioff()

In [ ]:
def plot_sections(label_file):
    labels = pd.read_parquet(label_file).astype({"spatialleiden": "category"})

    g = sns.FacetGrid(labels, col="brain_section_label", col_wrap=7, legend_out=False)
    _ = g.map_dataframe(
        sns.scatterplot,
        x="x",
        y="y",
        hue="spatialleiden",
        legend=False,
        **scatter_kwargs,
    )

    for ax in g.axes.flat:
        ax.set(aspect=1)
        remove_tick_and_label(ax)

    return g

## No integration

In [ ]:
fig = plot_sections(results_abc / "no_batch_integration.parquet")
fig.savefig(results_abc / "no_batch_integration.png")

## Harmony

In [ ]:
fig = plot_sections(results_abc / "harmony.parquet")
fig.savefig(results_abc / "harmony.png")

## scVI

In [ ]:
fig = plot_sections(results_abc / "scvi.parquet")
fig.savefig(results_abc / "scvi.png")